# Restricted Boltzmann Machines & Deep Belief Nets — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## energy-based binary latent variables
Log-linear models, sigmoid probabilities, and Markov chains are the prerequisites: an RBM uses an energy score, conditional Bernoulli updates, and sampling to learn hidden causes. This historical path leads directly to modern energy-based models (10.10) and to the idea that generation can be sampling from a learned landscape.

In [ ]:
# 1) enumerate visible-hidden energies for a tiny RBM
W=np.array([[.6,-.2],[.1,.4]]); b=np.array([.1,-.1]); c=np.array([0.,.2])
states=[]; energies=[]
for v in itertools.product([0,1], repeat=2):
  for h in itertools.product([0,1], repeat=2):
    v=np.array(v); h=np.array(h); states.append(''.join(map(str,v))+','+''.join(map(str,h))); energies.append(-(b@v+c@h+v@W@h))
plt.figure(figsize=(6,3)); plt.bar(range(len(energies)),energies); plt.title('RBM assigns an energy to each pair')
assert any(abs(e+0.7)<1e-9 for e in energies)

In [ ]:
# 2) unnormalized probabilities favor low energy
w=np.exp(-np.array(energies)); p=w/w.sum()
plt.figure(figsize=(6,3)); plt.bar(range(len(p)),p); plt.title('probability after normalizing exp(-E)')
assert abs(p.sum()-1)<1e-12

In [ ]:
# 3) hidden conditionals factorize given visible units
v=np.array([1,0]); logits=c+v@W; ph=1/(1+np.exp(-logits))
plt.figure(figsize=(4,3)); plt.bar(['h1','h2'],ph); plt.ylim(0,1); plt.title('p(h_j=1 | v)')
assert ph.shape==(2,) and ph[0]>0.5

In [ ]:
# 4) one Gibbs-like alternating update moves through states
rng=np.random.default_rng(1); v=np.array([1,0]); traj=[]
for _ in range(8):
  ph=1/(1+np.exp(-(c+v@W))); h=(rng.random(2)<ph).astype(int); pv=1/(1+np.exp(-(b+W@h))); v=(rng.random(2)<pv).astype(int); traj.append(v.copy())
plt.figure(figsize=(5,3)); plt.imshow(np.array(traj),aspect='auto',cmap='Greys'); plt.title('alternating visible samples')
assert np.array(traj).shape==(8,2)

In [ ]:
# 5) stacking RBMs forms deeper binary features
H=np.array([[0,0],[1,0],[0,1],[1,1]]); W2=np.array([[.5,-.3],[.2,.7]]); top=1/(1+np.exp(-(H@W2)))
plt.figure(figsize=(4,4)); plt.scatter(top[:,0],top[:,1],c=np.arange(4)); plt.title('second layer transforms hidden codes')
assert top.min()>0 and top.max()<1